# Bài 3  · Đánh giá phân loại
PGS.TS. Nguyễn Văn Hậu

Mở notebook trong thư mục Bai03. Cài requirements.txt nếu môi trường chưa có. Dữ liệu đã có sẵn nên thực hành không cần mạng. Đây là tái lập case lịch sử đã công bố; không lựa chọn lại bằng test.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'source/metrics_lab.py').exists():
    raise RuntimeError('Mở notebook trong thư mục Bai03 chứa source/ và data/.')
sys.path.insert(0,str(ROOT/'source'))
from metrics_lab import (run,measure,toy_data,predict_votes,select_policy,
    np,pd,confusion_matrix,roc_auc_score,average_precision_score)
report=run()
print('Kiểm khoa học:',report['status'],report['checks_count'])
print('Kết quả test lịch sử:',report['base'])


## 1. Một hàng thật trước khi đọc chỉ số
Nhãn y là đáp án nguồn; score là tỷ lệ phiếu, prediction là quyết định ở m6.


In [ ]:
test=pd.read_csv(ROOT/'data/test_predictions.csv')
validation=pd.read_csv(ROOT/'data/validation_predictions.csv')
print('Test:',test.y.value_counts().to_dict())
print('Validation:',validation.y.value_counts().to_dict())
for record in [1056,1501]:
    row=test.loc[test.source_record==record].iloc[0]
    print(record,repr(row.text))
    print('y / votes / score / prediction:',row.y,row.votes,row.score,row.prediction)


## 2. Tự đếm và dùng đúng nhóm mẫu số
Hàng thật, cột dự đoán; spam là lớp dương. Tính tay trước khi đối chiếu.


In [ ]:
toy=toy_data()
for tau in [.5,.8]:
    pred=(toy.score>=tau).astype(int)
    groups={name:toy.loc[(toy.y==y)&(pred==p),'id'].tolist()
        for name,y,p in [('TN',0,0),('FP',0,1),('FN',1,0),('TP',1,1)]}
    print('tau=',tau,'IDs=',groups)
    print(measure(toy.y,pred))
assert measure(toy.y,toy.score>=.5)['f1']==.8


## 3. Precision không có bảo đảm đơn điệu
Bỏ một TP khỏi nhóm cảnh báo có thể làm precision giảm.


In [ ]:
for tau in [.85,.9]:
    pred=toy.score>=tau
    print(tau,toy.loc[pred,'id'].tolist(),'P=',measure(toy.y,pred)['precision'])
assert measure(toy.y,toy.score>=.85)['precision']>measure(toy.y,toy.score>=.9)['precision']
edge=measure([0,0],[0,0])
print('Chỉ có TN; giá trị báo cáo và cờ definedness:',edge)


## 4. Chọn trên validation, khóa chính sách
Hai chi phí là giả định; tập ứng viên gồm cả m0 và m12.


In [ ]:
scan=[dict(min_votes=m,**measure(validation.y,predict_votes(validation.votes,m))) for m in range(13)]
chosen={key:select_policy(scan,key) for key in ['cost_5fp_fn','cost_fp_5fn']}
for key,best in chosen.items():
    print(key,'chon m=',best['min_votes'],'chi phi validation=',best[key])
    print('Test lich su:',measure(test.y,predict_votes(test.votes,best['min_votes'])))
assert chosen['cost_5fp_fn']['min_votes']==8
assert chosen['cost_fp_5fn']['min_votes']==4


## 5. Theo dõi hai ID qua ba chính sách
Không đổi y,score hoặc mô hình trong phép so.


In [ ]:
for record in [1056,1501]:
    row=test.loc[test.source_record==record].iloc[0]
    for m in [4,6,8]:
        pred=int(row.votes>=m)
        outcome=['TN','FP','FN','TP'][2*int(row.y)+pred]
        print(record,'m',m,'y',row.y,'prediction',pred,outcome)


## 6. Score cho xếp hạng; nhãn quyết định cho bốn ô
AUC/AP từ cùng score giữ nguyên khi chỉ đổi ngưỡng.


In [ ]:
print('ROC-AUC(score):',roc_auc_score(test.y,test.score))
print('AP(score):',average_precision_score(test.y,test.score))
for m in [4,6,8]:
    print('m',m,measure(test.y,predict_votes(test.votes,m)))
try:
    from IPython.display import SVG,display
except ImportError:
    print('Trong Jupyter có thể xem SVG; mở source/science/roc.svg và pr.svg để xem ngoài notebook.')
else:
    display(SVG(filename=str(ROOT/'source/science/roc.svg')))
    display(SVG(filename=str(ROOT/'source/science/pr.svg')))


## 7. Phiếu cuối buổi
Tự tính trước: TP3,FP2,FN2,TN13. Giải thích vì sao hai mẫu số cùng 5 chưa có nghĩa cùng nhóm.


In [ ]:
exit_result=measure([1]*5+[0]*15,[1]*3+[0]*2+[1]*2+[0]*13)
print(exit_result)
assert exit_result['precision']==exit_result['recall']==exit_result['f1']==.6
print('Kết luận cần có mục tiêu, dữ liệu lựa chọn, lịch sử test và một giới hạn cụ thể.')
